# Notebook 17: Diagnostic Analysis of RFF + SH Failure

## Goal
Understand WHY RFF catastrophically failed with SH features (-7.96%) in Notebook 16.

## Hypotheses from CRITICAL_ANALYSIS_NB16.md
1. **Frequency Interference** (most likely): SH and RFF use incompatible frequency representations
2. **Dimensionality Curse**: 100D input × 256 neurons × 25 frequencies = overparameterized
3. **Input Statistics Mismatch**: RFF designed for normalized inputs, SH features have different distribution
4. **Optimization Difficulty**: Complex loss landscape with interdependent parameters

## Experiments
1. **Input normalization**: Standardize SH features before RFF
2. **Training dynamics**: Visualize loss curves, gradient norms
3. **Activation visualization**: Plot learned RFF shapes
4. **Feature analysis**: Examine SH feature statistics
5. **Learnable frequencies**: Try learnable vs fixed frequencies
6. **Longer training**: 500 epochs instead of 100

In [ ]:
# Setup
import os
import sys

if 'COLAB_GPU' in os.environ:
    !rm -rf sample_data .config satclip gpw_data 2>/dev/null
    !git clone https://github.com/1hamzaiqbal/satclip.git
    !pip install lightning torchgeo huggingface_hub rasterio --quiet
    sys.path.append('./satclip/satclip')
else:
    sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'satclip'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
GPW_DIR = './gpw_data'
os.makedirs(GPW_DIR, exist_ok=True)

print("Extracting GPW data...")
with zipfile.ZipFile('/content/drive/MyDrive/grad/learned_activations/dataverse_files.zip', 'r') as z:
    z.extractall(GPW_DIR)

with zipfile.ZipFile(f'{GPW_DIR}/gpw-v4-population-density-rev11_2020_15_min_tif.zip', 'r') as z:
    z.extractall(GPW_DIR)
print("Done!")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import time
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
from sklearn.metrics import r2_score
import positional_encoding as PE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

---
## Activation Classes

In [ ]:
# =============================================================================
# RFF ACTIVATION
# =============================================================================

class RFFActivation(nn.Module):
    """Random Fourier Features activation."""
    def __init__(self, n_features=25, max_freq=10.0, learnable_freq=False, freq_init='linear'):
        super().__init__()
        self.n_features = n_features
        self.learnable_freq = learnable_freq

        # Frequency initialization
        if freq_init == 'linear':
            freqs = torch.linspace(0.1, max_freq, n_features)
        elif freq_init == 'log':
            freqs = torch.logspace(-2, np.log10(max_freq), n_features)
        elif freq_init == 'random':
            freqs = torch.rand(n_features) * max_freq
        else:
            raise ValueError(f"Unknown freq_init: {freq_init}")

        if learnable_freq:
            self.freqs = nn.Parameter(freqs)
        else:
            self.register_buffer('freqs', freqs)

        self.sin_coeff = nn.Parameter(torch.randn(n_features) * 0.1)
        self.cos_coeff = nn.Parameter(torch.randn(n_features) * 0.1)
        self.scale = nn.Parameter(torch.ones(1))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        wx = x.unsqueeze(-1) * self.freqs
        sin_part = (torch.sin(wx) * self.sin_coeff).sum(-1)
        cos_part = (torch.cos(wx) * self.cos_coeff).sum(-1)
        return self.scale * (sin_part + cos_part) + self.bias


# =============================================================================
# SPLINE ACTIVATION
# =============================================================================

class SplineActivation(nn.Module):
    """Cubic B-spline activation with learnable control points."""
    def __init__(self, n_knots=10, input_range=(-3.0, 3.0), init='relu'):
        super().__init__()
        self.n_knots = n_knots
        self.input_range = input_range

        knot_x = torch.linspace(input_range[0], input_range[1], n_knots)
        self.register_buffer('knot_x', knot_x)

        if init == 'relu':
            knot_y = torch.relu(knot_x)
        elif init == 'linear':
            knot_y = knot_x.clone()
        elif init == 'zero':
            knot_y = torch.zeros(n_knots)
        else:
            knot_y = torch.randn(n_knots) * 0.1

        self.knot_y = nn.Parameter(knot_y)

    def forward(self, x):
        x_clamped = torch.clamp(x, self.input_range[0], self.input_range[1])
        x_norm = (x_clamped - self.input_range[0]) / (self.input_range[1] - self.input_range[0])
        x_idx = x_norm * (self.n_knots - 1)

        idx_low = torch.floor(x_idx).long()
        idx_high = torch.clamp(idx_low + 1, max=self.n_knots - 1)
        idx_low = torch.clamp(idx_low, max=self.n_knots - 1)

        weight = x_idx - idx_low.float()
        y_low = self.knot_y[idx_low]
        y_high = self.knot_y[idx_high]

        return y_low + weight * (y_high - y_low)


# =============================================================================
# SIREN ACTIVATION
# =============================================================================

class SirenLayer(nn.Module):
    """SIREN layer from SatCLIP."""
    def __init__(self, dim_in, dim_out, w0=1.0, is_first=False):
        super().__init__()
        self.dim_in = dim_in
        self.w0 = w0
        self.is_first = is_first

        self.linear = nn.Linear(dim_in, dim_out)
        self._init_weights()

    def _init_weights(self):
        if self.is_first:
            bound = 1.0 / self.dim_in
        else:
            bound = np.sqrt(6.0 / self.dim_in) / self.w0

        self.linear.weight.data.uniform_(-bound, bound)
        if self.linear.bias is not None:
            self.linear.bias.data.uniform_(-bound, bound)

    def forward(self, x):
        return torch.sin(self.w0 * self.linear(x))

print("Activation classes loaded!")

---
## Universal Encoder

In [ ]:
class UniversalEncoder(nn.Module):
    """
    Universal encoder that supports:
    - Input: Raw coords (2D) or SH features
    - Activation: RFF, Spline, SIREN, or ReLU
    - Optional: Input normalization for SH features
    """
    def __init__(self, input_type='raw', sh_legendre_polys=None,
                 activation_type='rff', activation_kwargs=None,
                 n_layers=3, hidden_dim=256, output_dim=256,
                 normalize_sh=False, sh_mean=None, sh_std=None):
        super().__init__()
        self.input_type = input_type
        self.activation_type = activation_type
        self.normalize_sh = normalize_sh

        # Input encoding
        if input_type == 'raw':
            self.posenc = None
            input_dim = 2
        elif input_type == 'sh':
            assert sh_legendre_polys is not None
            self.posenc = PE.SphericalHarmonics(
                legendre_polys=sh_legendre_polys,
                harmonics_calculation='analytic'
            )
            with torch.no_grad():
                test_coords = torch.zeros(1, 2)
                test_output = self.posenc(test_coords)
                input_dim = test_output.shape[1]
            print(f"  SH(L={sh_legendre_polys}) output dim: {input_dim}")
        else:
            raise ValueError(f"Unknown input_type: {input_type}")

        # Store normalization params
        if normalize_sh and sh_mean is not None and sh_std is not None:
            self.register_buffer('sh_mean', torch.tensor(sh_mean, dtype=torch.float32))
            self.register_buffer('sh_std', torch.tensor(sh_std, dtype=torch.float32))
        else:
            self.sh_mean = None
            self.sh_std = None

        if activation_kwargs is None:
            activation_kwargs = {}

        # Build network
        dims = [input_dim] + [hidden_dim] * n_layers + [output_dim]

        if activation_type == 'siren':
            self.layers = nn.ModuleList()
            for i in range(len(dims) - 1):
                is_first = (i == 0)
                w0 = 30.0 if is_first else 1.0
                if i < len(dims) - 2:
                    self.layers.append(SirenLayer(dims[i], dims[i+1], w0=w0, is_first=is_first))
                else:
                    linear = nn.Linear(dims[i], dims[i+1])
                    bound = np.sqrt(6.0 / dims[i]) / 1.0
                    linear.weight.data.uniform_(-bound, bound)
                    if linear.bias is not None:
                        linear.bias.data.uniform_(-bound, bound)
                    self.layers.append(linear)
            self.activations = None

        elif activation_type in ['rff', 'spline']:
            self.linears = nn.ModuleList([
                nn.Linear(dims[i], dims[i+1])
                for i in range(len(dims) - 1)
            ])

            act_class = RFFActivation if activation_type == 'rff' else SplineActivation
            self.activations = nn.ModuleList([
                act_class(**activation_kwargs)
                for _ in range(n_layers)
            ])

            for linear in self.linears:
                nn.init.kaiming_normal_(linear.weight)
                nn.init.zeros_(linear.bias)

        elif activation_type == 'relu':
            layers = []
            for i in range(len(dims) - 1):
                layers.append(nn.Linear(dims[i], dims[i+1]))
                if i < len(dims) - 2:
                    layers.append(nn.ReLU())
            self.net = nn.Sequential(*layers)
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.kaiming_normal_(m.weight)
                    nn.init.zeros_(m.bias)

        else:
            raise ValueError(f"Unknown activation_type: {activation_type}")

    def forward(self, coords):
        # Input encoding
        if self.input_type == 'raw':
            x = coords / torch.tensor([180., 90.], device=coords.device)
        else:  # 'sh'
            x = self.posenc(coords)
            # Apply normalization if enabled
            if self.normalize_sh and self.sh_mean is not None and self.sh_std is not None:
                x = (x - self.sh_mean) / self.sh_std

        # Forward through network
        if self.activation_type == 'siren':
            for layer in self.layers:
                x = layer(x)
        elif self.activation_type in ['rff', 'spline']:
            for i, (linear, act) in enumerate(zip(self.linears[:-1], self.activations)):
                x = act(linear(x))
            x = self.linears[-1](x)
        else:  # 'relu'
            x = self.net(x)

        return x

print("Universal encoder loaded!")

---
## Data Loading

In [ ]:
# Load population data
Image.MAX_IMAGE_PIXELS = None

print("Loading population data...")
img = Image.open(f'{GPW_DIR}/gpw_v4_population_density_rev11_2020_15_min.tif')
pop_data = np.array(img)
h, w = pop_data.shape
lons = np.linspace(-180 + 180/w, 180 - 180/w, w)
lats = np.linspace(90 - 90/h, -90 + 90/h, h)
print(f"Shape: {pop_data.shape}")

# Spatial blocking (same as notebook 16)
def sample_blocked(data, lons, lats, n_samples=15000, grid_size=5.0, test_ratio=0.3, seed=42):
    np.random.seed(seed)
    valid = data > -1e30

    n_lon = int(360 / grid_size)
    n_lat = int(180 / grid_size)
    n_cells = n_lon * n_lat

    test_cells = set(np.random.choice(n_cells, int(n_cells * test_ratio), replace=False))

    valid_idx = np.where(valid)
    n_valid = len(valid_idx[0])
    sample_idx = np.random.choice(n_valid, min(n_samples, n_valid), replace=False)

    rows, cols = valid_idx[0][sample_idx], valid_idx[1][sample_idx]
    sample_lons, sample_lats = lons[cols], lats[rows]
    sample_vals = data[rows, cols]

    train_mask = []
    for lon, lat in zip(sample_lons, sample_lats):
        cell = int((lat + 90) / grid_size) * n_lon + int((lon + 180) / grid_size)
        cell = min(cell, n_cells - 1)
        train_mask.append(cell not in test_cells)
    train_mask = np.array(train_mask)

    coords = np.stack([sample_lons, sample_lats], axis=1)
    return coords[train_mask], sample_vals[train_mask], coords[~train_mask], sample_vals[~train_mask]

coords_train, vals_train, coords_test, vals_test = sample_blocked(pop_data, lons, lats)
print(f"Train: {len(coords_train)}, Test: {len(coords_test)}")

---
## 1. Analyze SH Feature Statistics

In [ ]:
# Generate SH features for training data
print("Computing SH features for training data...")
L = 10
posenc = PE.SphericalHarmonics(legendre_polys=L, harmonics_calculation='analytic')

coords_train_tensor = torch.from_numpy(coords_train).float()
sh_features = posenc(coords_train_tensor).numpy()

print(f"SH features shape: {sh_features.shape}")
print(f"\nSH Feature Statistics:")
print(f"  Per-feature mean (first 5): {sh_features.mean(axis=0)[:5]}")
print(f"  Per-feature std (first 5):  {sh_features.std(axis=0)[:5]}")
print(f"  Per-feature min (first 5):  {sh_features.min(axis=0)[:5]}")
print(f"  Per-feature max (first 5):  {sh_features.max(axis=0)[:5]}")
print(f"\nOverall:")
print(f"  Global mean: {sh_features.mean():.4f}")
print(f"  Global std: {sh_features.std():.4f}")
print(f"  Global min: {sh_features.min():.4f}")
print(f"  Global max: {sh_features.max():.4f}")

# Compute normalization statistics
sh_mean = sh_features.mean(axis=0)
sh_std = sh_features.std(axis=0) + 1e-8
print(f"\nNormalization computed: mean={sh_mean.mean():.4f}, std={sh_std.mean():.4f}")

In [ ]:
# Visualize SH feature distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('SH Feature Distributions (L=10, 100 features)', fontsize=14)

for idx, ax in enumerate(axes.flat):
    ax.hist(sh_features[:, idx], bins=50, alpha=0.7, edgecolor='black')
    ax.set_title(f'Feature {idx}')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.axvline(sh_features[:, idx].mean(), color='red', linestyle='--', label='Mean')
    ax.legend()

plt.tight_layout()
plt.show()

print("\nKey observations:")
print(f"- Features are NOT normalized (not zero-mean, unit-variance)")
print(f"- Features have different scales across dimensions")
print(f"- This violates RFF's assumption of normalized inputs!")

---
## 2. Training Function with Metrics

In [ ]:
class PopulationPredictor(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1)
        )

    def forward(self, coords):
        return self.head(self.encoder(coords)).squeeze(-1)


def train_and_evaluate(name, encoder, coords_train, vals_train, coords_test, vals_test,
                      epochs=100, batch_size=256, lr=1e-3, verbose=True):
    """Train encoder and track detailed metrics."""
    model = PopulationPredictor(encoder).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    train_X = torch.tensor(coords_train, dtype=torch.float32)
    train_y = torch.tensor(np.log1p(vals_train), dtype=torch.float32)
    test_X = torch.tensor(coords_test, dtype=torch.float32).to(device)
    test_y = torch.tensor(np.log1p(vals_test), dtype=torch.float32)

    loader = DataLoader(TensorDataset(train_X, train_y),
                       batch_size=batch_size, shuffle=True)

    train_losses = []
    test_r2s = []
    grad_norms = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        epoch_grad_norm = 0.0

        for X, y in loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(X), y)
            loss.backward()

            # Track gradient norm
            total_norm = 0.0
            for p in model.parameters():
                if p.grad is not None:
                    total_norm += p.grad.data.norm(2).item() ** 2
            total_norm = total_norm ** 0.5
            epoch_grad_norm += total_norm

            opt.step()
            epoch_loss += loss.item()

        train_losses.append(epoch_loss / len(loader))
        grad_norms.append(epoch_grad_norm / len(loader))

        # Evaluate every 10 epochs
        if (epoch + 1) % 10 == 0 or epoch == 0:
            model.eval()
            with torch.no_grad():
                pred = model(test_X).cpu().numpy()
            r2 = r2_score(test_y.numpy(), pred)
            test_r2s.append(r2)

            if verbose:
                print(f"Epoch {epoch+1}/{epochs}: Loss={train_losses[-1]:.6f}, R²={r2:.4f}, GradNorm={grad_norms[-1]:.4f}")

    return {
        'model': name,
        'train_losses': train_losses,
        'test_r2s': test_r2s,
        'grad_norms': grad_norms,
        'final_r2': test_r2s[-1],
        'trained_model': model
    }

print("Training functions loaded!")

---
## 3. Experiment 1: Input Normalization Effect

In [ ]:
print("="*80)
print("EXPERIMENT 1: Does normalization fix RFF + SH failure?")
print("="*80)

results_dict = {}

In [ ]:
# Test 1: SH + RFF without normalization (reproduce failure)
print("\n[1/5] SH + RFF (no normalization) - Reproducing the failure...")
enc = UniversalEncoder(
    input_type='sh',
    sh_legendre_polys=10,
    activation_type='rff',
    activation_kwargs={'n_features': 25},
    normalize_sh=False
)
results_dict['rff_no_norm'] = train_and_evaluate(
    'SH + RFF (no norm)', enc, coords_train, vals_train, coords_test, vals_test
)
print(f"Final R²: {results_dict['rff_no_norm']['final_r2']:.4f}")

In [ ]:
# Test 2: SH + RFF WITH normalization
print("\n[2/5] SH + RFF (WITH normalization) - Does this fix it?...")
enc = UniversalEncoder(
    input_type='sh',
    sh_legendre_polys=10,
    activation_type='rff',
    activation_kwargs={'n_features': 25},
    normalize_sh=True,
    sh_mean=sh_mean,
    sh_std=sh_std
)
results_dict['rff_norm'] = train_and_evaluate(
    'SH + RFF (norm)', enc, coords_train, vals_train, coords_test, vals_test
)
print(f"Final R²: {results_dict['rff_norm']['final_r2']:.4f}")
improvement = results_dict['rff_norm']['final_r2'] - results_dict['rff_no_norm']['final_r2']
print(f"Improvement: {improvement:.4f} ({100*improvement/results_dict['rff_no_norm']['final_r2']:.2f}%)")

In [ ]:
# Test 3: SH + ReLU baseline (should work well)
print("\n[3/5] SH + ReLU (baseline) - Should be ~0.749...")
enc = UniversalEncoder(
    input_type='sh',
    sh_legendre_polys=10,
    activation_type='relu'
)
results_dict['relu'] = train_and_evaluate(
    'SH + ReLU', enc, coords_train, vals_train, coords_test, vals_test
)
print(f"Final R²: {results_dict['relu']['final_r2']:.4f}")

In [ ]:
# Test 4: SH + SIREN baseline
print("\n[4/5] SH + SIREN (baseline) - Should be ~0.743...")
enc = UniversalEncoder(
    input_type='sh',
    sh_legendre_polys=10,
    activation_type='siren'
)
results_dict['siren'] = train_and_evaluate(
    'SH + SIREN', enc, coords_train, vals_train, coords_test, vals_test
)
print(f"Final R²: {results_dict['siren']['final_r2']:.4f}")

In [ ]:
# Test 5: SH + Spline (should work)
print("\n[5/5] SH + Spline (k=10) - Should be ~0.748...")
enc = UniversalEncoder(
    input_type='sh',
    sh_legendre_polys=10,
    activation_type='spline',
    activation_kwargs={'n_knots': 10}
)
results_dict['spline'] = train_and_evaluate(
    'SH + Spline (k=10)', enc, coords_train, vals_train, coords_test, vals_test
)
print(f"Final R²: {results_dict['spline']['final_r2']:.4f}")

---
## 4. Visualize Training Dynamics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Training loss
axes[0].plot(results_dict['rff_no_norm']['train_losses'], label='RFF (no norm)', alpha=0.7)
axes[0].plot(results_dict['rff_norm']['train_losses'], label='RFF (norm)', alpha=0.7)
axes[0].plot(results_dict['relu']['train_losses'], label='ReLU', alpha=0.7)
axes[0].plot(results_dict['spline']['train_losses'], label='Spline', alpha=0.7)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Test R²
test_epochs = list(range(0, 100, 10)) + [99]
axes[1].plot(test_epochs[:len(results_dict['rff_no_norm']['test_r2s'])], 
             results_dict['rff_no_norm']['test_r2s'], 'o-', label='RFF (no norm)', alpha=0.7)
axes[1].plot(test_epochs[:len(results_dict['rff_norm']['test_r2s'])], 
             results_dict['rff_norm']['test_r2s'], 'o-', label='RFF (norm)', alpha=0.7)
axes[1].plot(test_epochs[:len(results_dict['relu']['test_r2s'])], 
             results_dict['relu']['test_r2s'], 'o-', label='ReLU', alpha=0.7)
axes[1].plot(test_epochs[:len(results_dict['spline']['test_r2s'])], 
             results_dict['spline']['test_r2s'], 'o-', label='Spline', alpha=0.7)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test R²')
axes[1].set_title('Test R² Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Gradient norms
axes[2].plot(results_dict['rff_no_norm']['grad_norms'], label='RFF (no norm)', alpha=0.7)
axes[2].plot(results_dict['rff_norm']['grad_norms'], label='RFF (norm)', alpha=0.7)
axes[2].plot(results_dict['relu']['grad_norms'], label='ReLU', alpha=0.7)
axes[2].plot(results_dict['spline']['grad_norms'], label='Spline', alpha=0.7)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Gradient Norm')
axes[2].set_title('Gradient Norms Over Time')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].set_yscale('log')

plt.tight_layout()
plt.show()

print("\nGradient Norm Analysis:")
for key, name in [('rff_no_norm', 'RFF (no norm)'), ('rff_norm', 'RFF (norm)'), 
                  ('relu', 'ReLU'), ('spline', 'Spline')]:
    print(f"  {name:20s}: final grad norm = {results_dict[key]['grad_norms'][-1]:.4f}")

---
## 5. Results Summary Table

In [ ]:
# Create results table
summary_data = []
baseline_r2 = results_dict['siren']['final_r2']

for key, name in [('siren', 'SH + SIREN'), 
                  ('relu', 'SH + ReLU'),
                  ('spline', 'SH + Spline'),
                  ('rff_no_norm', 'SH + RFF (no norm)'),
                  ('rff_norm', 'SH + RFF (norm)')]:
    r2 = results_dict[key]['final_r2']
    summary_data.append({
        'Model': name,
        'R²': r2,
        'vs SIREN': r2 - baseline_r2,
        '% vs SIREN': 100 * (r2 - baseline_r2) / baseline_r2
    })

df_summary = pd.DataFrame(summary_data)

print("\n" + "="*80)
print("DIAGNOSTIC RESULTS SUMMARY")
print("="*80)
print(df_summary.to_string(index=False))
print("="*80)

# Save results
df_summary.to_csv('diagnostic_results_nb17.csv', index=False)
print("\nResults saved to diagnostic_results_nb17.csv")

---
## 6. Experiment 2: Learnable Frequencies

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 2: Do learnable frequencies help?")
print("="*80)

enc = UniversalEncoder(
    input_type='sh',
    sh_legendre_polys=10,
    activation_type='rff',
    activation_kwargs={'n_features': 25, 'learnable_freq': True},
    normalize_sh=True,
    sh_mean=sh_mean,
    sh_std=sh_std
)

results_dict['rff_learnable'] = train_and_evaluate(
    'SH + RFF (learnable)', enc, coords_train, vals_train, coords_test, vals_test
)

print(f"\nFinal R² (learnable freq): {results_dict['rff_learnable']['final_r2']:.4f}")
print(f"vs fixed freq: {results_dict['rff_norm']['final_r2']:.4f}")
diff = results_dict['rff_learnable']['final_r2'] - results_dict['rff_norm']['final_r2']
print(f"Difference: {diff:+.4f}")

---
## 7. Conclusions

### Key Findings:

1. **Did normalization fix the failure?**
   - Without normalization: R² ≈ 0.66 (catastrophic)
   - With normalization: R² ≈ ??? (check results above)
   - **Verdict**: [To be determined]

2. **Training dynamics:**
   - Check gradient norms: Are they stable or exploding?
   - Check convergence: Does RFF converge slower than ReLU/Spline?

3. **Learnable frequencies:**
   - Does learning frequencies improve over fixed?
   - **Verdict**: [To be determined]

4. **Comparison to baselines:**
   - ReLU: ~0.749 (best overall)
   - Spline: ~0.748 (+0.56% vs SIREN)
   - SIREN: ~0.743 (baseline)
   - RFF normalized: ??? (check if it matches or beats SIREN)

### Next Steps:

**If normalization fixes it (RFF norm R² > 0.74)**:
- ✅ Update all SH + RFF experiments to use normalized features
- ✅ Proceed with Phase 2 of roadmap
- Test longer training (500 epochs)

**If normalization doesn't fix it (RFF norm R² < 0.72)**:
- ❌ Frequency interference hypothesis confirmed
- ❌ Avoid SH + RFF combinations
- ✅ Focus on SH + Spline instead
- ✅ Raw + RFF might still work

**If learnable frequencies help significantly**:
- Make learnable_freq=True the default
- Test on other tasks

See `DIAGNOSTIC_CONCLUSIONS_NB17.md` for detailed writeup.